#  Egyptian Movies Dataset - Preprocessing


## Preprocessing Steps:
1. **Load Data**: Import CSV file
2. **Lowercase Conversion**: Standardize text
3. **Tokenization**: Split text into words
4. **Stopword Removal**: Remove common words
5. **Stemming**: Apply Snowball stemmers
6. **Save Results**: Export preprocessed data


## Import Required Libraries

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

## Download NLTK Resources

In [2]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

## Configuration

In [3]:
INPUT_FILE="/content/movies_tv_all_sources.csv"
OUTPUT_FILE="../output/movies_preprocessed.csv"
TEXT_COLUMNS=['title', 'abstract']

## Step 1: Load Data

In [4]:
df=pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} movies")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Loaded 10042 movies

Columns: ['doc_id', 'title', 'type', 'year', 'rating', 'votes', 'popularity', 'abstract', 'source']

First few rows:


,doc_id,title,type,year,rating,votes,popularity,abstract,source
0,doc_1,7 Dogs,movie,2026.0,0.000,0,11.2325,Interpol agent Khalid Al-Azzazi joins forces w...,tmdb
1,doc_2,Karmouz War,movie,2018.0,5.942,52,5.6905,"Alexandria, Egypt, 1940. Three young Egyptians...",tmdb
2,doc_3,EgyBest,movie,2026.0,0.000,0,3.0636,Two friends dive into the digital underworld c...,tmdb
3,doc_4,Love Story,movie,2019.0,7.500,18,2.7851,"Youssef, whose brother is trying to push him t...",tmdb
4,doc_5,Incident of Dishonor,movie,1971.0,0.000,0,1.3651,One of the beauties of the estate lives with h...,tmdb


In [5]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDataset Info:")
df.info()

Missing values per column:
doc_id          0
title           0
type            0
year          215
rating         14
votes          12
popularity     39
abstract        0
source          0
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10042 entries, 0 to 10041
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   doc_id      10042 non-null  object 
 1   title       10042 non-null  object 
 2   type        10042 non-null  object 
 3   year        9827 non-null   float64
 4   rating      10028 non-null  float64
 5   votes       10030 non-null  object 
 6   popularity  10003 non-null  float64
 7   abstract    10042 non-null  object 
 8   source      10042 non-null  object 
dtypes: float64(3), object(6)
memory usage: 706.2+ KB


## Step 2: Combine Text Fields


In [6]:
def combine_text_fields(row):
    text_parts=[]
    if pd.notna(row['title']):
        text_parts.append(str(row['title']))
    if pd.notna(row['abstract']):
        text_parts.append(str(row['abstract']))
    return ' '.join(text_parts)
df['combined_text']=df.apply(combine_text_fields, axis=1)

#small example
print("\nSample combined text:")
print(df['combined_text'].iloc[0])


Sample combined text:
7 Dogs Interpol agent Khalid Al-Azzazi joins forces with Ghali Abu Dawood from the 7 Dogs crime syndicate to fight drug trafficking.


## Step 3: Lowercase Conversion

In [7]:
lowercase_text=[]
for text in df['combined_text']:
    if pd.isna(text):
        lowercase_text.append("")
    else:
        lowercase_text.append(str(text).lower())
df['lowercase_text']=lowercase_text

#small example
print("Before:")
print(df['combined_text'].iloc[0])
print("After:")
print(df['lowercase_text'].iloc[0])

Before:
7 Dogs Interpol agent Khalid Al-Azzazi joins forces with Ghali Abu Dawood from the 7 Dogs crime syndicate to fight drug trafficking.
After:
7 dogs interpol agent khalid al-azzazi joins forces with ghali abu dawood from the 7 dogs crime syndicate to fight drug trafficking.


## Step 4: Tokenization (Split into words)

In [8]:
tokens_list=[]
for text in df['lowercase_text']:
    # Use regex to extract words (alphanumeric)
    words=re.findall(r'\w+', text)
    tokens_list.append(words)
df['tokens']=tokens_list

#small example
print(f"\nSample tokens (first 20):")
print(df['tokens'].iloc[0][:20])


Sample tokens (first 20):
['7', 'dogs', 'interpol', 'agent', 'khalid', 'al', 'azzazi', 'joins', 'forces', 'with', 'ghali', 'abu', 'dawood', 'from', 'the', '7', 'dogs', 'crime', 'syndicate', 'to']


## Step 5: Stopword Removal

In [9]:
stop_words=set(stopwords.words('english'))
print(f"Number of stopwords: {len(stop_words)}")

no_stopwords_list=[]
for tokens in df['tokens']:
    filtered_words=[word for word in tokens if word.lower() not in stop_words]
    no_stopwords_list.append(filtered_words)

df['no_stopwords']=no_stopwords_list

#small example
print(f"\nBefore (with stopwords): {len(df['tokens'].iloc[0])} words")
print(df['tokens'].iloc[0][:15])
print(f"\nAfter (no stopwords): {len(df['no_stopwords'].iloc[0])} words")
print(df['no_stopwords'].iloc[0][:15])

Number of stopwords: 198

Before (with stopwords): 23 words
['7', 'dogs', 'interpol', 'agent', 'khalid', 'al', 'azzazi', 'joins', 'forces', 'with', 'ghali', 'abu', 'dawood', 'from', 'the']

After (no stopwords): 19 words
['7', 'dogs', 'interpol', 'agent', 'khalid', 'al', 'azzazi', 'joins', 'forces', 'ghali', 'abu', 'dawood', '7', 'dogs', 'crime']


## Step 6: Stemming


In [10]:
snowball=SnowballStemmer('english')
snowball_stemmed_list=[]

for words in df['no_stopwords']:
    snowball_words=[snowball.stem(word) for word in words]
    snowball_stemmed_list.append(snowball_words)
df['snowball_stem']=snowball_stemmed_list

## Step 7: Create Final Processed Text


In [11]:
processed_text_list=[]
for stemmed_words in df['snowball_stem']:
    joined_text=' '.join(stemmed_words)
    processed_text_list.append(joined_text)

df['processed_text']=processed_text_list

#small example
print(f"Original: {df['combined_text'].iloc[0][:100]}...")
print(f"\nProcessed: {df['processed_text'].iloc[0][:100]}...")

Original: 7 Dogs Interpol agent Khalid Al-Azzazi joins forces with Ghali Abu Dawood from the 7 Dogs crime synd...

Processed: 7 dog interpol agent khalid al azzazi join forc ghali abu dawood 7 dog crime syndic fight drug traff...


## Step 8: Prepare Output Dataset

In [12]:
output_df=df[[
    'doc_id',
    'title',
    'type',
    'year',
    'rating',
    'abstract',
    'processed_text'
]].copy()

output_df.rename(columns={
    'doc_id': 'doc_id',
    'abstract': 'original_abstract'
}, inplace=True)

#small example
print(f"\nOutput columns: {list(output_df.columns)}")
print(f"\nFirst 3 rows:")
output_df.head(3)


Output columns: ['doc_id', 'title', 'type', 'year', 'rating', 'original_abstract', 'processed_text']

First 3 rows:


,doc_id,title,type,year,rating,original_abstract,processed_text
0,doc_1,7 Dogs,movie,2026.0,0.000,Interpol agent Khalid Al-Azzazi joins forces w...,7 dog interpol agent khalid al azzazi join for...
1,doc_2,Karmouz War,movie,2018.0,5.942,"Alexandria, Egypt, 1940. Three young Egyptians...",karmouz war alexandria egypt 1940 three young ...
2,doc_3,EgyBest,movie,2026.0,0.000,Two friends dive into the digital underworld c...,egybest two friend dive digit underworld chase...


## Step 9: Save Preprocessed Data

In [13]:
import os
output_dir=os.path.dirname(OUTPUT_FILE)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# Save to CSV
output_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"Total records: {len(output_df)}")

Created directory: ../output
Total records: 10042


In [15]:
#from google.colab import files
#files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>